In [1]:
import time
start_time=time.time()

In [2]:
import numpy as np
import pandas as pd
import sys
from pathlib import Path
import astropy.units as u
from astropy.io import fits

sys.path.append(str(Path().resolve() / 'py_modules')) # 1 level up = project root
from turb_utils import make_extended, make_3dfield

In [3]:
line        = 'Synthetic 2D map'
lambda_rest = 0
line_id     = 'Synthetic 2D map'
line_id2    = "Nan"

In [4]:
# Physical parameters
# Data from observations
bins        = 0

region     = 'Synthetic 2D map'
name_reg   = 'Synthetic 2D map'
reg_id     = 'Synthetic 2D map'
inst       = 'NaN'
inst_id    = 'NaN'
type_obj   = 'NaN'

pix        = 1  # Instrument resolution [arcsec/pix]
dist       = 1 # Distance [parsecs]
s0_        = 0.0    # proposed seeing
pc         = 1
#pc         = dist*(2*np.pi) / (360 * 60 * 60) # value in parsecs of each arcsec
s0         = 0
#s0         = (s0_*pc) / 2.355    # RMS seeing [parsecs]
noise      = 0

# Fake map parameters
N     = 256
ra    = N / 256 # Artificial correlation length
m_obs = 1.0    # The projected slope of the structure function we measure in nature
m2D   = 0.85   # for recovering m as 1.00 in the projected field (k = 2D + m2D) non emissivity fluctuation case
m3D_1 = 0.30   # for recovering m as 1.00 in the projected field (k = 3D + m3D) light fluctuations
m3D_2 = 0.55   # for recovering m as 1.00 in the projected field (k = 3D + m3D) heavy fluctuations
ellip      = 0.5
theta      = 45
randomseed_vv = 1894_04_25
print(ra)


1.0


In [5]:
#Since we are working with matrix data binning can be applied
#name_file    =  line_id2 
#name_id      =  inst_id + '-' + reg_id + '-' + line_id + '-' + 'bin' + str(bins)

name      = 'fake_map_mod_finite_m' +str(int((2+m2D)*100))+'_r' +str(int(ra)) \
+ '_N' + str(N) + '_bin' + str(bins)
name_id   = name
name_file = name
print(name)

fake_map_mod_finite_m285_r1_N256_bin0


In [6]:
# Tapered map
vmap = make_extended(
    N,
    powerlaw           = 2.0 + m2D,
    ellip              = ellip,
    theta              = theta,
    correlation_length = ra,
    randomseed         = randomseed_vv,
)

sig = vmap.std()
vmap /= sig

sig

0.22274608717846364

# Export

In [7]:
primary = fits.PrimaryHDU()
primary.header["FILE"]       = (name_file, "Fits file name")
primary.header["NAME_ID"]    = (name_id, "pipeline identifier")
primary.header["REG"]        = (region, "Region catalog name")
primary.header["NAME_REG"]   = (name_reg, "Region name")
primary.header["REG_ID"]     = (reg_id, "Region identifier")
primary.header["TYPE_OBJ"]   = (type_obj, "Type object")
primary.header["DIST"]       = (dist, "Distance in parsecs")
primary.header["INSTR"]      = (inst, "Instrument")
primary.header["LINE"]       = (line, "Emission line name")
primary.header["LINE_ID"]    = (line_id, "Emission line identifier")
primary.header["L_REST"]     = (lambda_rest, "Emission line name")
primary.header["SIG"]        = (sig, "Standard deviation vel map")
primary.header["SIG2"]       = (sig**2, "Vel Variance map")
primary.header["S0"]         = (s0, "Atmospheric seeing")
primary.header["BOX_SIZE"]   = (N, "Observational box_size")
primary.header["PC"]         = (pc* (2**bins), "parsec convertion")
primary.header["PIX"]        = (pix, "Instrument pixel scale")
primary.header["BINSIZE"]    = (bins, "Spatial binning factor")
primary.header["noise"]      = (noise, "Instrumental noise")
primary.header["m2D"]       = (m2D, "k = 3 + m3D")
#primary.header["m3D"]       = (m3D, "k = 3 + m3D")
primary.header["ra"]         = (ra, "Artificial correlation length")
primary.header["SEED1"]      = (randomseed_vv, "Random seed velocity")
#primary.header["SEED2"]     = (randomseed_sb, "Random seed surfavebrightness")
primary.header["AUTHOR"]     = ("J. Garcia-Vazquez", "File creator")

In [8]:
#hdu_m0 = fits.ImageHDU(data=sb_m,  name='MOM0')
hdu_m1 = fits.ImageHDU(data=vmap,  name='MOM1')
#hdu_m2 = fits.ImageHDU(data=ss_m[trim],  name='MOM2')
#hdu_mk = fits.ImageHDU(data=good.astype(np.uint8) ,  name='MASK')

In [9]:
hdul = fits.HDUList([primary, hdu_m1])
hdul.writeto('fits_ready/' + name_id + '.fits', overwrite=True)

In [10]:
with fits.open("fits_ready/" + name_id + ".fits") as hdul:
    hdul.info()
    
hdr = hdul[0].header
print(hdr)

Filename: fits_ready/fake_map_mod_finite_m285_r1_N256_bin0.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU      27   ()      
  1  MOM1          1 ImageHDU         8   (256, 256)   float64   
SIMPLE  =                    T / conforms to FITS standard                      BITPIX  =                    8 / array data type                                NAXIS   =                    0 / number of array dimensions                     EXTEND  =                    T                                                  FILE    = 'fake_map_mod_finite_m285_r1_N256_bin0' / Fits file name              NAME_ID = 'fake_map_mod_finite_m285_r1_N256_bin0' / pipeline identifier         REG     = 'Synthetic 2D map'   / Region catalog name                            NAME_REG= 'Synthetic 2D map'   / Region name                                    REG_ID  = 'Synthetic 2D map'   / Region identifier                              TYPE_OBJ= 'NaN     '           / Type

In [11]:
print("--- %s seconds ---" % (time.time()-start_time))

--- 1.7475552558898926 seconds ---
